# AeroHealth: Turbofan Engine RUL Prediction (NASA C-MAPSS FD001)

Predicts **Remaining Useful Life (RUL)** from sensor windows using Linear/Ridge, Random Forest, LSTM, GRU and **GRU + Temporal Attention**.

Design choices (and why):
- **Engine-wise split** (no leakage); feature selection and scaling use *training engines only*.
- **RUL capped at 125** (standard piece-wise linear target). Predictions are clipped to [0, 125].
- **Fair baselines:** classical models are run on both the last time-step and the full flattened 30-cycle window.
- **Multiple seeds** for every neural model, reported as mean ± std, with a paired GRU vs GRU+Attention comparison.
- **Test set = official FD001 test** (last 30 cycles of each of the 100 engines) and is the headline number.
  Validation is reported two ways: all windows (what early-stopping sees) and last-window-per-engine (comparable to test).

In [ ]:
import os, random, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, LSTM, Layer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

SEED, WINDOW, RUL_CAP = 42, 30, 125     # SEED fixes the engine split + classical models
N_SEEDS, EPOCHS = 5, 60                 # neural-net seeds (3 models x N_SEEDS trainings)
MODEL_SEEDS = [SEED + i for i in range(N_SEEDS)]

def set_seed(s): tf.keras.utils.set_random_seed(s)   # seeds python, numpy and tensorflow
set_seed(SEED)

os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"Linear Regression": "#9CA3AF", "Ridge (window)": "#6B7280", "Random Forest": "#10B981",
          "Random Forest (window)": "#047857", "LSTM": "#F59E0B", "GRU": "#2563EB", "GRU + Attention": "#DC2626"}
NN = ["LSTM", "GRU", "GRU + Attention"]
def save(name): plt.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight")
print("TF", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

## 1. Data

In [ ]:
DATA_DIR = "CMAPSSData"; os.makedirs(DATA_DIR, exist_ok=True)
MIRRORS = ["https://raw.githubusercontent.com/hankroark/Turbofan-Engine-Degradation/master/CMAPSSData/",
           "https://raw.githubusercontent.com/mapr-demos/predictive-maintenance/master/notebooks/jupyter/Dataset/CMAPSSData/"]
for f in ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]:
    p = f"{DATA_DIR}/{f}"
    for base in MIRRORS:
        if os.path.exists(p): break
        try: urllib.request.urlretrieve(base + f, p)
        except Exception as ex: print("mirror failed:", ex)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Could not download {f}. Upload it manually to ./{DATA_DIR}/")

cols = ["engine_id", "cycle"] + [f"op_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
def load(p):
    df = pd.read_csv(p, sep=r"\s+", header=None).dropna(axis=1, how="all")
    df.columns = cols
    return df

train_raw = load(f"{DATA_DIR}/train_FD001.txt")
test_raw  = load(f"{DATA_DIR}/test_FD001.txt")
test_rul  = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["rul"])["rul"].values
print("train:", train_raw.shape, "| test:", test_raw.shape, "| test RUL labels:", len(test_rul))

## 2. Preprocessing: labels, engine-wise split, feature selection and scaling (train only), windows

In [ ]:
# Capped piece-wise linear RUL label (computed per engine, so no cross-engine leakage)
train_raw["RUL"] = (train_raw.groupby("engine_id")["cycle"].transform("max") - train_raw["cycle"]).clip(upper=RUL_CAP)

# Engine-wise 80/20 split
ids = train_raw["engine_id"].unique().copy()
np.random.default_rng(SEED).shuffle(ids)
tr_ids, va_ids = ids[:80], ids[80:]
tr = train_raw[train_raw.engine_id.isin(tr_ids)].copy()
va = train_raw[train_raw.engine_id.isin(va_ids)].copy()

# Feature selection on TRAIN engines only: drop (near-)constant channels
base_cols = [f"op_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
feat_cols = [c for c in base_cols if tr[c].var() > 1e-5]
print(f"{len(feat_cols)} input features kept")

# Scale (fit on training engines only)
scaler = StandardScaler().fit(tr[feat_cols].astype("float64"))
def scaled(df):
    df = df.copy(); df[feat_cols] = scaler.transform(df[feat_cols].astype("float64")); return df
tr, va, te = scaled(tr), scaled(va), scaled(test_raw)

def make_windows(df, w=WINDOW):
    X, y, meta = [], [], []
    for eid, g in df.groupby("engine_id"):
        g = g.sort_values("cycle"); f, t, c = g[feat_cols].values, g["RUL"].values, g["cycle"].values
        for e in range(w, len(g) + 1):
            X.append(f[e-w:e]); y.append(t[e-1]); meta.append((eid, c[e-1]))
    return np.array(X, np.float32), np.array(y, np.float32), pd.DataFrame(meta, columns=["engine_id", "cycle"])

X_tr, y_tr, _       = make_windows(tr)
X_va, y_va, va_meta = make_windows(va)

# Official test set: last WINDOW cycles of every test engine
test_groups = [g.sort_values("cycle") for _, g in te.groupby("engine_id")]
assert all(len(g) >= WINDOW for g in test_groups), "a test engine is shorter than the window"
X_te = np.stack([g[feat_cols].values[-WINDOW:] for g in test_groups]).astype(np.float32)
y_te = np.minimum(test_rul, RUL_CAP).astype(np.float32)
assert len(X_te) == len(y_te) == 100

# Index of the last window of each validation engine (comparable to how the test set is scored)
va_last = va_meta.groupby("engine_id")["cycle"].idxmax().values
print("train", X_tr.shape, "| val", X_va.shape, "| val-last", len(va_last), "| test", X_te.shape)

## 3. Metrics

In [ ]:
def nasa_score(y, p):                      # asymmetric: late predictions (p > y) are penalised more
    d = p - y
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

def metrics(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": r2_score(y, p), "NASA": nasa_score(y, p)}

rows = []
def log(model, seed, pv, pt):
    r = {"Model": model, "Seed": seed}
    for k, v in metrics(y_va, pv).items():              r[f"Val {k}"] = v
    for k, v in metrics(y_va[va_last], pv[va_last]).items(): r[f"ValLast {k}"] = v
    for k, v in metrics(y_te, pt).items():              r[f"Test {k}"] = v
    rows.append(r)

## 4. Classical baselines (last time-step *and* full flattened window, for a fair comparison)

In [ ]:
preds_va, preds_te = {}, {}     # seed-0 predictions, used for the figures
clip = lambda p: np.clip(p, 0, RUL_CAP)
flat = lambda X: X.reshape(len(X), -1)

baselines = {
    "Linear Regression":      (LinearRegression(),                                              lambda X: X[:, -1, :]),
    "Ridge (window)":         (Ridge(alpha=10.0),                                               flat),
    "Random Forest":          (RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1), lambda X: X[:, -1, :]),
    "Random Forest (window)": (RandomForestRegressor(n_estimators=200, max_features="sqrt", random_state=SEED, n_jobs=-1), flat),
}
for name, (m, f) in baselines.items():
    m.fit(f(X_tr), y_tr)
    pv, pt = clip(m.predict(f(X_va))), clip(m.predict(f(X_te)))
    preds_va[name], preds_te[name] = pv, pt
    log(name, SEED, pv, pt)
    print(f"{name:24s} test RMSE {metrics(y_te, pt)['RMSE']:.2f}")

## 5. Neural models (LSTM, GRU, GRU + Temporal Attention), trained over several seeds

In [ ]:
class TemporalAttention(Layer):
    """Additive attention over time. attn_weights(x) returns the (batch, time) weights."""
    def build(self, shape):
        h = int(shape[-1])
        self.W = self.add_weight(name="W", shape=(h, h), initializer="glorot_uniform")
        self.b = self.add_weight(name="b", shape=(h,), initializer="zeros")
        self.u = self.add_weight(name="u", shape=(h,), initializer="glorot_uniform")
        super().build(shape)
    def attn_weights(self, x):
        e = tf.tensordot(tf.tanh(tf.tensordot(x, self.W, 1) + self.b), self.u, 1)
        return tf.nn.softmax(e, axis=1)
    def call(self, x):
        return tf.reduce_sum(x * tf.expand_dims(self.attn_weights(x), -1), axis=1)
    def compute_output_shape(self, shape): return (shape[0], shape[-1])

def build_model(kind, seed):
    set_seed(seed)
    inp = Input((WINDOW, X_tr.shape[2]))
    if kind == "GRU + Attention":
        x = Dropout(0.2)(GRU(64, return_sequences=True)(inp))
        x = Dropout(0.2, name="pre_attention")(GRU(32, return_sequences=True)(x))
        x = TemporalAttention(name="attention")(x)
    else:
        R = LSTM if kind == "LSTM" else GRU
        x = Dropout(0.2)(R(64, return_sequences=True)(inp))
        x = Dropout(0.2)(R(32)(x))
    out = Dense(1)(Dense(32, activation="relu")(x))
    m = Model(inp, out, name=kind.replace(" + ", "_").replace(" ", "_"))
    m.compile(Adam(1e-3), loss="mse", metrics=["mae"])
    return m

models0, histories = {}, {}            # seed-0 models / histories, used for figures
for kind in NN:
    for i, s in enumerate(MODEL_SEEDS):
        m = build_model(kind, s)
        h = m.fit(X_tr, y_tr / RUL_CAP, validation_data=(X_va, y_va / RUL_CAP),   # target scaled to [0, 1]
                  epochs=EPOCHS, batch_size=128, verbose=0,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
                             ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)])
        pv = clip(m.predict(X_va, verbose=0).ravel() * RUL_CAP)
        pt = clip(m.predict(X_te, verbose=0).ravel() * RUL_CAP)
        log(kind, s, pv, pt)
        print(f"{kind:16s} seed {s} | epochs {len(h.history['loss']):2d} | test RMSE {metrics(y_te, pt)['RMSE']:.2f}")
        if i == 0:
            models0[kind], histories[kind] = m, h.history
            preds_va[kind], preds_te[kind] = pv, pt

## 6. Results (mean ± std over seeds for neural models; classical models are deterministic)

In [ ]:
raw = pd.DataFrame(rows)
raw.to_csv("figures/results_per_seed.csv", index=False)

metric_cols = [c for c in raw.columns if c not in ("Model", "Seed")]
agg = raw.groupby("Model", sort=False)[metric_cols].agg(["mean", "std"])
agg.columns = [f"{a} {b}" for a, b in agg.columns]
agg.round(3).to_csv("figures/results_table_numeric.csv")

def fmt(model, col):
    m, s = agg.loc[model, f"{col} mean"], agg.loc[model, f"{col} std"]
    return f"{m:.2f}" if np.isnan(s) else f"{m:.2f} ± {s:.2f}"
show_cols = ["Test MAE", "Test RMSE", "Test R2", "Test NASA", "ValLast RMSE", "Val RMSE"]
results = pd.DataFrame({c: [fmt(m, c) for m in agg.index] for c in show_cols}, index=agg.index)
results.to_csv("figures/results_table.csv")
display(results)

# Paired comparison on identical seeds: is attention really better than plain GRU?
g = raw[raw.Model == "GRU"].set_index("Seed"); a = raw[raw.Model == "GRU + Attention"].set_index("Seed")
for col in ["Test RMSE", "Test MAE", "Test NASA"]:
    d = (a[col] - g[col])
    print(f"Attention - GRU, {col:10s}: mean {d.mean():+.2f}, std {d.std():.2f}, attention better in {(d < 0).sum()}/{len(d)} seeds")

## 7. Figures (figures 3-6 use the seed-0 models)

In [ ]:
# Fig 1: test performance, mean +/- std over seeds
order = list(agg.index)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (title, col) in zip(axes, [("MAE (cycles)", "Test MAE"), ("RMSE (cycles)", "Test RMSE"), ("R$^2$", "Test R2")]):
    mu, sd = agg[f"{col} mean"].values, agg[f"{col} std"].fillna(0).values
    ax.bar(range(len(order)), mu, yerr=sd, capsize=3, color=[COLORS[m] for m in order])
    for i, v in enumerate(mu): ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=30, ha="right"); ax.set_title(title)
plt.suptitle(f"Official FD001 test set (neural models: mean ± std over {N_SEEDS} seeds)", fontsize=13)
plt.tight_layout(); save("fig1_model_comparison"); plt.show()

# Fig 2: validation loss (seed 0), in cycles^2
plt.figure(figsize=(7, 4))
for n in NN: plt.plot(np.array(histories[n]["val_loss"]) * RUL_CAP**2, label=n, color=COLORS[n], lw=2)
plt.yscale("log"); plt.xlabel("Epoch"); plt.ylabel("Validation MSE (cycles$^2$, log)")
plt.title("Validation loss during training (seed 0, best epoch restored)"); plt.legend()
plt.tight_layout(); save("fig2_training_curves"); plt.show()

# Fig 3: predicted vs actual (test, one dot per engine). Dots at Actual = 125 are the RUL cap.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, n in zip(axes, NN):
    ax.scatter(y_te, preds_te[n], s=22, alpha=0.7, color=COLORS[n])
    ax.plot([0, RUL_CAP], [0, RUL_CAP], "k--", lw=1)
    ax.set_title(f"{n}  (RMSE {metrics(y_te, preds_te[n])['RMSE']:.1f})"); ax.set_xlabel("Actual RUL (capped at 125)"); ax.set_ylabel("Predicted RUL")
plt.suptitle("Test set: predicted vs actual RUL (seed 0)", fontsize=13); plt.tight_layout(); save("fig3_parity_test"); plt.show()

# Fig 4: RUL trajectory of one validation engine
eid = va_meta.engine_id.iloc[0]; mask = (va_meta.engine_id == eid).values
cyc = va_meta.cycle[mask]
plt.figure(figsize=(8, 4))
plt.plot(cyc, y_va[mask], "k", lw=2.5, label="Actual (capped)")
for n in NN: plt.plot(cyc, preds_va[n][mask], color=COLORS[n], lw=1.6, label=n)
plt.xlabel("Flight cycle"); plt.ylabel("RUL (cycles)"); plt.title(f"RUL trajectory - validation engine {eid} (seed 0)")
plt.legend(); plt.tight_layout(); save("fig4_rul_trajectory"); plt.show()

# Fig 5: test error distribution, one panel per model (same bins)
bins = np.linspace(-60, 60, 25)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, n in zip(axes, NN):
    e = preds_te[n] - y_te
    ax.hist(e, bins=bins, color=COLORS[n], alpha=0.85); ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(f"{n}  (mean error {e.mean():+.1f})"); ax.set_xlabel("Predicted - actual (cycles)")
axes[0].set_ylabel("Engines"); plt.suptitle("Test set error distribution (seed 0)", fontsize=13)
plt.tight_layout(); save("fig5_error_distribution"); plt.show()

# Fig 6: attention weights (seed-0 GRU + Attention)
att = models0["GRU + Attention"]
feat_model = Model(att.input, att.get_layer("pre_attention").output)
h = feat_model.predict(X_te, verbose=0)
w = att.get_layer("attention").attn_weights(h).numpy()          # (engines, WINDOW)
assert np.allclose(w.sum(1), 1, atol=1e-4)
srt = np.argsort(y_te)                                          # sort engines by true RUL
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.4]})
a1.bar(range(-WINDOW + 1, 1), w.mean(0), color=COLORS["GRU + Attention"])
a1.set_xlabel("Cycle relative to prediction time"); a1.set_ylabel("Mean attention weight"); a1.set_title("Average attention over window")
im = a2.imshow(w[srt], aspect="auto", cmap="viridis", extent=[-WINDOW + 1, 0, len(srt), 0])
a2.set_xlabel("Cycle relative to prediction time"); a2.set_ylabel("Test engines (low RUL -> high RUL)"); a2.set_title("Attention heatmap")
plt.colorbar(im, ax=a2, label="Weight"); a2.grid(False)
plt.tight_layout(); save("fig6_attention"); plt.show()

# Fig 7: seed-to-seed spread of test RMSE (is attention's gain bigger than noise?)
plt.figure(figsize=(6, 4))
for i, n in enumerate(NN):
    v = raw[raw.Model == n]["Test RMSE"].values
    plt.scatter(np.full(len(v), i) + np.linspace(-0.08, 0.08, len(v)), v, color=COLORS[n], s=40, zorder=3)
    plt.hlines(v.mean(), i - 0.2, i + 0.2, color="k", lw=2)
plt.xticks(range(len(NN)), NN); plt.ylabel("Test RMSE (cycles)"); plt.title(f"Seed-to-seed variability ({N_SEEDS} seeds; bar = mean)")
plt.tight_layout(); save("fig7_seed_variability"); plt.show()

## 8. Predicted remaining cycles per test engine

In [ ]:
pred_table = pd.DataFrame({"engine": np.arange(1, len(y_te) + 1), "actual_RUL(capped)": y_te})
for n in NN: pred_table[f"{n} pred"] = preds_te[n].round(1)
pred_table.to_csv("figures/test_predictions.csv", index=False)
display(pred_table.head(10))

## 9. (Colab) download all outputs

In [ ]:
import shutil
shutil.make_archive("aerohealth_figures", "zip", "figures")
try:
    from google.colab import files; files.download("aerohealth_figures.zip")
except Exception:
    print("Saved: aerohealth_figures.zip")